# EDA - House Prices
Análisis exploratorio del dataset de entrenamiento (`data/train.csv`), variable objetivo `SalePrice`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

TARGET = "SalePrice"
ID_COL = "Id"

df = pd.read_csv("data/train.csv")
df.head()

## Dimensiones y tipos de variables

In [ ]:
print(f"filas: {df.shape[0]}, columnas: {df.shape[1]}")
df.dtypes.value_counts()

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.drop([ID_COL, TARGET])
cat_cols = df.select_dtypes(exclude=np.number).columns

print(f"numéricas: {len(num_cols)}")
print(f"categóricas: {len(cat_cols)}")
print(list(cat_cols))

## Estadísticas descriptivas

In [ ]:
df[TARGET].describe()

In [ ]:
df[num_cols].describe().T

In [ ]:
df[cat_cols].describe().T

## Valores nulos

In [ ]:
nulls = df.isnull().sum()
nulls = nulls[nulls > 0].sort_values(ascending=False)
null_pct = (nulls / len(df) * 100).round(2)
pd.DataFrame({"n_nulos": nulls, "pct": null_pct})

In [ ]:
plt.figure(figsize=(10, 8))
sns.barplot(y=nulls.index, x=nulls.values)
plt.title("Valores nulos por columna")
plt.xlabel("cantidad de nulos")
plt.tight_layout()
plt.show()

**Nota:** varias columnas categóricas (`Alley`, `PoolQC`, `Fence`, `FireplaceQu`, `GarageType`, etc.) tienen nulos que en el diccionario de datos original representan "sin esa característica" (ej. `Alley = NaN` significa que la casa no tiene callejón), no un dato faltante aleatorio. Antes de imputar, conviene revisar caso por caso si corresponde imputar con la moda o con una categoría explícita tipo `"None"`.

## Distribución de la variable objetivo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df[TARGET], kde=True, ax=axes[0])
axes[0].set_title("Distribución de SalePrice")
sns.boxplot(x=df[TARGET], ax=axes[1])
axes[1].set_title("Boxplot de SalePrice")
plt.tight_layout()
plt.show()

print("skewness:", df[TARGET].skew())

`SalePrice` suele venir sesgada a la derecha; si el sesgo es alto, considerar una transformación logarítmica (`np.log1p`) para el entrenamiento.

## Outliers en variables numéricas

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16, 14))
for ax, col in zip(axes.flatten(), num_cols[:16]):
    sns.boxplot(x=df[col], ax=ax)
    ax.set_title(col, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
def iqr_outliers(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((series < lower) | (series > upper)).sum()

outlier_counts = df[num_cols].apply(iqr_outliers).sort_values(ascending=False)
outlier_counts[outlier_counts > 0]

## Correlaciones entre variables numéricas

In [ ]:
corr = df[list(num_cols) + [TARGET]].corr()

plt.figure(figsize=(14, 12))
sns.heatmap(corr, cmap="coolwarm", center=0, square=True, linewidths=0.3)
plt.title("Matriz de correlación")
plt.tight_layout()
plt.show()

In [ ]:
top_corr = corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
top_corr.head(15)

## Relación de las variables numéricas más correlacionadas con SalePrice

In [ ]:
top_features = top_corr.head(6).index

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flatten(), top_features):
    sns.scatterplot(x=df[col], y=df[TARGET], ax=ax, alpha=0.5)
    ax.set_title(f"{col} vs {TARGET}")
plt.tight_layout()
plt.show()

## Variables categóricas vs SalePrice

In [ ]:
cat_to_plot = ["Neighborhood", "OverallQual", "HouseStyle", "SaleCondition"]
cat_to_plot = [c for c in cat_to_plot if c in df.columns]

fig, axes = plt.subplots(len(cat_to_plot), 1, figsize=(12, 5 * len(cat_to_plot)))
if len(cat_to_plot) == 1:
    axes = [axes]
for ax, col in zip(axes, cat_to_plot):
    order = df.groupby(col)[TARGET].median().sort_values(ascending=False).index
    sns.boxplot(x=col, y=TARGET, data=df, order=order, ax=ax)
    ax.set_title(f"{col} vs {TARGET}")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## Cardinalidad de variables categóricas

In [ ]:
cardinality = df[cat_cols].nunique().sort_values(ascending=False)
cardinality

**Nota:** columnas con alta cardinalidad (ej. `Neighborhood`) generan muchas columnas al aplicar one-hot encoding; vale la pena evaluar si conviene agrupar categorías poco frecuentes.

## Decisiones de preprocesamiento derivadas del EDA

- **Nulos categóricos** que representan "sin característica" (`Alley`, `PoolQC`, `Fence`, `FireplaceQu`, `GarageType`, `GarageFinish`, `GarageQual`, `GarageCond`, `BsmtQual`, `BsmtCond`, `BsmtExposure`, `BsmtFinType1`, `BsmtFinType2`, `MasVnrType`): imputar con una categoría explícita `"None"` en vez de la moda.
- **Nulos numéricos** (`LotFrontage`, `GarageYrBlt`, `MasVnrArea`): imputar con mediana o 0 según el caso (`GarageYrBlt` nulo probablemente significa que no hay garage).
- **Variable objetivo sesgada**: evaluar `np.log1p(SalePrice)` como target de entrenamiento, revirtiendo con `np.expm1` al predecir.
- **Outliers extremos** en `GrLivArea` u otras variables con alta correlación: revisar y considerar remover casos atípicos conocidos del dataset (ej. casas muy grandes con precio bajo).
- **Escalado**: `StandardScaler` para numéricas, tal como ya hace el pipeline de `train.py`.
- **Categóricas de alta cardinalidad**: revisar si conviene agrupar categorías poco frecuentes antes del one-hot encoding para no explotar la dimensionalidad.